# Worksheet 2: building a network with `torch.nn`

Your project: *teaching a neural network to play Atari Pong using nothing but the
screen.*

Worksheet 1 showed the problem: 100,800 numbers in, one of two actions out. This builds
the thing in the middle.

Nothing here plays Pong or learns anything. Every input is fake, random numbers of the
right shape, and that is deliberate: it lets you check the *shapes* before there is any
chance of mistaking a shape bug for a learning bug.

## 1. What you will learn

1. How to read a network diagram and write the same thing in PyTorch.
2. The shape rules, and how to read a shape error instead of guessing at it.
3. The standard way to define a network: subclass, register layers, implement `forward`.
4. Why the *same* network can be a Pong player, a chess player or a Go player, and what
   that tells you about where the real work is.
5. What dropout, ReLU variants, normalisation and weight initialisation each do, seen in
   output rather than described.
6. How to save a network and load it back, and what a checkpoint does not contain.

## Running this notebook

Locally: the environment comes from the repo. `uv sync --locked`, then
`uv run jupyter lab`. The cell below does nothing.

On Colab: the kernel exists before the project does, so the cell below clones the
repo and installs what Colab is missing. Run it first.

It installs `gymnasium` and `ale-py` only. Colab already provides a working CUDA build
of torch, and replacing it with the exact locked one means downloading the entire CUDA
stack for no benefit to anything taught here. The README describes the stricter route
if you ever need bit-identical versions.

The cell ends by importing `pong`, so if anything above failed you find out here rather
than three cells later.


In [1]:
# Colab bootstrap. Does nothing when you run this notebook locally.
import sys

IN_COLAB = "google.colab" in sys.modules
REPO = "/content/pong"
URL = "https://github.com/piyushahuja/pong.git"

if IN_COLAB:
    # Idempotent: clone if it is missing, refresh it if it is already there.
    # Safe to re-run, and fast on a runtime that has done it before.
    ![ -d {REPO} ] || git clone -q {URL} {REPO}
    !git -C {REPO} pull -q || true
    %cd {REPO}

    !command -v uv >/dev/null || curl -LsSf https://astral.sh/uv/install.sh | env UV_INSTALL_DIR=/usr/local/bin sh
    !uv pip install -q --system --python {sys.executable} gymnasium ale-py

    # Deliberately not an editable install: that works by dropping a .pth file
    # into site-packages, and .pth files are only read when the interpreter
    # starts, so it would not take effect in this already-running kernel.
    sys.path.insert(0, f"{REPO}/src")

import pong        # fails here, loudly, if any of the above did not work

print("python:", sys.version.split()[0])
print("pong package from:", pong.PROJECT_ROOT)


python: 3.12.14
pong package from: /Users/piyush/Code/ml/pong


In [2]:
# Confirm the environment is what you think it is.
import torch
import gymnasium
import ale_py

print("torch:     ", torch.__version__)
print("gymnasium: ", gymnasium.__version__)
print("ale-py:    ", ale_py.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


torch:      2.14.0
gymnasium:  1.3.0
ale-py:     0.12.1
CUDA available: False


The line that matters is the restored forward pass producing the same shape from a network
built fresh and loaded from disk.

Two things about that. First, `state_dict()` saves the weights, not the code. To load
it back you had to construct `TwoLayerNet(8, 16, 4)` yourself, with exactly those numbers.
Get them wrong and the load fails, which is why a checkpoint is only meaningful next to
the architecture it was trained with.

Second, `map_location=device` is what lets a checkpoint trained on one device load on
another; without it a file saved from a GPU refuses to load on a CPU-only machine. And
`weights_only=True` is the safe default: it loads tensors and refuses to execute code
embedded in the file.

If that printed versions without complaint, the environment is right and nothing in this
worksheet needs the emulator, only `torch`.

In [3]:
import math
from pathlib import Path

import torch
import torch.nn as nn

print(torch.__version__)
import pong        # PROJECT_ROOT: paths that work from anywhere


2.14.0


## 2. What a two-layer net looks like

![Two-layer network: inputs to hidden units to one output](../assets/nn-two-layer.png)

Read the picture as code:

| In the figure | In PyTorch |
|---|---|
| Left (many inputs) | `input_dim`, length of the input vector |
| Blue circles | `hidden_dim`, `nn.Linear(input_dim, hidden_dim)` | 
| Pink circle | `output_dim`, here `1`, often via `nn.Linear(hidden_dim, 1)` |
| Lines | learnable weights inside those `Linear` layers |

If the single output is passed through a sigmoid, it becomes a number in $(0,1)$.

Blue circles are neurons. Each neuron is connected to each coordinate of the input vector by
its own weight, so the number of weights is `input_dim` × `hidden_dim`.

Another way to see this: the input vector and the neurons define a matrix of shape
`(hidden_dim, input_dim)`, where each **row** corresponds to a neuron and each **column** to a
coordinate of the input vector.

Those two readings are the same object. Hover a **neuron** to light its row, an **input
coordinate** to light its column, or a single **cell** to light the one edge it is. Move the
sliders to watch the weight count track `input_dim` × `hidden_dim`.

Note the transpose, which catches everyone once: you write `nn.Linear(input_dim, hidden_dim)`,
but the tensor it stores is `.weight` of shape `(hidden_dim, input_dim)`.

In [4]:
import html

from IPython.display import HTML

# Shown in an iframe rather than injected straight into the page. The demo is a
# complete HTML document, and its <style> sets rules on `body`; dropped into a
# notebook output those would restyle the whole notebook. srcdoc keeps it sealed
# off, and embedding the text rather than pointing at a path means it works the
# same locally and on Colab.
demo = (pong.PROJECT_ROOT / "demos" / "weights-as-matrix.html").read_text()

HTML(f'<iframe srcdoc="{html.escape(demo, quote=True)}" '
     'style="width:100%; height:560px; border:1px solid #ddd; border-radius:4px;"></iframe>')


/Users/piyush/Code/ml/pong/.venv/lib/python3.12/site-packages/IPython/core/display.py:448: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


### See it on a real frame

That matrix view is a diagram. The same idea, running on a trained policy and a real rally,
is here:

[Open the forward pass demo](https://piyushahuja.com/pong/demos/forward-pass.html)

The court carries the difference between consecutive frames, which is the only thing the
policy ever sees. The 200 bars are the hidden units firing. Come back once you have watched
a point being won; worksheet 4 takes it apart properly.

## 3. Tensors and shapes

`nn.Linear(in_features, out_features)` is one linear layer of neurons that takes in input featues and outputs outpur features.


- `(features,)`, one example, no batch axis
- `(1, features)`, batch of size 1
- `(B, features)`, batch of size `B`

`unsqueeze` inserts a size-1 dimension. `squeeze` removes size-1 dimensions.

In [5]:
features = 8
x = torch.randn(features)
print("vector:", x.shape)

x_batched = x.unsqueeze(0)          # add a batch dim at front
print("unsqueeze(0):", x_batched.shape)

x_col = x.unsqueeze(-1)             # add a dim at the end
print("unsqueeze(-1):", x_col.shape)

print("squeeze back:", x_batched.squeeze(0).shape)
print("squeeze(-1):", x_col.squeeze(-1).shape)

vector: torch.Size([8])
unsqueeze(0): torch.Size([1, 8])
unsqueeze(-1): torch.Size([8, 1])
squeeze back: torch.Size([8])
squeeze(-1): torch.Size([8])


Compare `unsqueeze(0)` giving `(1, 8)` with `unsqueeze(-1)` giving `(8, 1)`. Both added one
dimension, both hold the same eight numbers, and they mean completely different things to a
`Linear` layer.

`(1, 8)` is one example with eight features, and `Linear(8, ...)` accepts it. `(8, 1)` is
eight examples each with one feature, and the same layer rejects it, because its last
dimension is 1 rather than 8.

This is the most common shape mistake in PyTorch, which is why the next cell causes it on
purpose.

In [6]:
linear = nn.Linear(in_features=8, out_features=3)

ok = torch.randn(8)
print("ok input", ok.shape, "->", linear(ok).shape)

ok_batch = torch.randn(4, 8)
print("ok batch", ok_batch.shape, "->", linear(ok_batch).shape)

# Deliberate mistake: last dim is 7, but Linear expects 8
try:
    linear(torch.randn(7))
except RuntimeError as e:
    print("shape error (expected):")
    print(e)

ok input torch.Size([8]) -> torch.Size([3])
ok batch torch.Size([4, 8]) -> torch.Size([4, 3])
shape error (expected):
mat1 and mat2 shapes cannot be multiplied (1x7 and 8x3)


Three things in that output.

The unbatched input `(8,)` became `(3,)`, and the batch `(4, 8)` became `(4, 3)`, the batch
dimension passed straight through. A `Linear` layer does not care how many examples you
hand it.

Then read the error properly rather than skimming it. It names both shapes and says which
multiplication failed. Once you can map "mat1 and mat2 shapes cannot be multiplied" onto
"my last dimension was 7 and the layer wanted 8", a shape error becomes a two-second fix.

And note that it is a `RuntimeError` at call time, not an error when the layer was built.
PyTorch checks shapes when data flows through, not when you define the network.

## 4. Define, instantiate, call

1. Subclass `nn.Module`
2. Call `super().__init__()`
3. Register layers as attributes (`self.layer1 = ...`)
4. Implement `forward`
5. Instantiate with `model = Net(...)`
6. Call with `y = model(x)` (this runs `forward`)

In [7]:
class TinyNet(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.layer1 = nn.Linear(input_dim, hidden_dim)
        self.layer2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = self.layer1(x)
        x = nn.functional.relu(x)
        x = self.layer2(x)
        return x


model = TinyNet(input_dim=8, hidden_dim=5, output_dim=3)
print(model)

y = model(torch.randn(8))
print("output shape:", y.shape)
print("output:", y)

TinyNet(
  (layer1): Linear(in_features=8, out_features=5, bias=True)
  (layer2): Linear(in_features=5, out_features=3, bias=True)
)
output shape: torch.Size([3])
output: tensor([ 0.4674,  0.4207, -0.8615], grad_fn=<ViewBackward0>)


In [8]:
# What the optimizer will actually see. This is the bit that matters.
for name, parameter in model.named_parameters():
    print(f"{name:15s} {tuple(parameter.shape)}  {parameter.numel()} numbers")

print("total:", sum(p.numel() for p in model.parameters()))

layer1.weight   (5, 8)  40 numbers
layer1.bias     (5,)  5 numbers
layer2.weight   (3, 5)  15 numbers
layer2.bias     (3,)  3 numbers
total: 63


Check the arithmetic yourself rather than taking the total on trust. `layer1.weight` is
`(5, 8)`, five neurons, eight inputs each, 40 numbers, plus `layer1.bias` with 5. Then
`layer2.weight` is `(3, 5)` and `layer2.bias` is 3. Total 63.

Notice we never wrote `self.relu = ...`. ReLU has no weights, so there is nothing to
register, and calling `F.relu` inside `forward` is enough.

That cuts both ways, and it is the bug worth knowing about: a layer that you forget to
assign to `self` will not appear here, so the optimizer will never update it. Training will
run, report a loss, and quietly fail to learn that part of the network.

## 5. A concrete two-layer network

Same pattern as the figure, with optional dropout between layers.

- Several outputs → often treated as logits. Softmax turns them into a distribution over options.
- One output + sigmoid → one number in $(0,1)$.

In [9]:
class TwoLayerNet(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, dropout=0.0):
        super().__init__()
        self.layer1 = nn.Linear(input_dim, hidden_dim)
        self.dropout = nn.Dropout(dropout)
        self.layer2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = self.layer1(x)
        x = self.dropout(x)
        x = nn.functional.relu(x)
        x = self.layer2(x)
        return x


# Many outputs → logits
multi = TwoLayerNet(input_dim=16, hidden_dim=8, output_dim=4, dropout=0.1)
logits = multi(torch.randn(16))
probs = nn.functional.softmax(logits, dim=-1)
print("logits", logits.shape, logits)
print("softmax", probs.shape, probs, "sum=", probs.sum().item())

# One output → optional sigmoid
single = TwoLayerNet(input_dim=16, hidden_dim=8, output_dim=1)
logit = single(torch.randn(16))
p = torch.sigmoid(logit).squeeze(-1)
print("logit", logit.shape, "-> sigmoid squeezed", p.shape, p.item())

logits torch.Size([4]) tensor([ 0.2698, -0.0936,  0.0640,  0.3694], grad_fn=<ViewBackward0>)
softmax torch.Size([4]) tensor([0.2767, 0.1924, 0.2252, 0.3057], grad_fn=<SoftmaxBackward0>) sum= 1.0
logit torch.Size([1]) -> sigmoid squeezed torch.Size([]) 0.5807536840438843


The raw logits are just numbers, some negative, no particular range. After softmax they
are all positive and sum to 1, which is what makes it legitimate to call them a
probability distribution.

The single-output case is the one Pong uses. `sigmoid` squashes one number into $(0,1)$, and
`.squeeze(-1)` drops the trailing size-1 dimension so you get a plain scalar rather than a
one-element tensor. Both matter: a scalar is what the Bernoulli distribution in Worksheet 3
expects.

## 6. Interpretation examples (shapes only)

A network is just numbers in → numbers out. You *can* choose `input_dim` / `output_dim` so those numbers are easy to *read as* preferences over moves in a game.

The cells below do not play a game or learn anything. They only show shapes and one forward pass on fake inputs.

### Pong-shaped outputs (matches the figure)

Flattened screen features in; one number out. After sigmoid, you *could* interpret it as “prefer UP” (high) vs “prefer DOWN” (low).

In [10]:
class PongShapedNet(nn.Module):
    def __init__(self, input_dim=6400, hidden_dim=32):
        super().__init__()
        self.layer1 = nn.Linear(input_dim, hidden_dim)
        self.layer2 = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        x = nn.functional.relu(self.layer1(x))
        x = torch.sigmoid(self.layer2(x))
        return x.squeeze(-1)


pong_net = PongShapedNet()
fake_frame = torch.randn(6400)          # stand-in for flattened pixels
prefer_up = pong_net(fake_frame)
print("input", fake_frame.shape)
print("output", prefer_up.shape, prefer_up.item(), "(interpret as prefer UP)")

input torch.Size([6400])
output torch.Size([]) 0.5749640464782715 (interpret as prefer UP)


In [11]:
print("parameters:", sum(p.numel() for p in pong_net.parameters()))
print("of which in layer 1:", pong_net.layer1.weight.numel())
print("\nat the project's real hidden size of 200:", 6400 * 200 + 200)

parameters: 204865
of which in layer 1: 204800

at the project's real hidden size of 200: 1280200


Where 6400 comes from: a Pong frame is 210 × 160 × 3, but the project crops away the
scoreboard, takes every second pixel in each direction and keeps one colour channel,
leaving an 80 × 80 grid that flattens to 6400. That was Exercise 1.2 in Worksheet 1.

Now look at where the parameters are. Almost every one is in the first layer, because the
input is enormous, and at the project's real `hidden=200` that is about 1.28 million
numbers in a network only two layers deep. The input size makes this network big, not its
depth.

And the output is one number between 0 and 1. Calling it "probability of UP" is a decision
we make about it. The network would be equally content if we called it "probability of
rain".

### Chess-shaped outputs

Board features in; many outputs. Softmax → a distribution over move slots. Here we use a toy size (`64`) so the demo stays small; a fuller encoding of chess moves would use a much larger `output_dim`.

In [12]:
BOARD_FEATURES = 64          # toy: one number per square
MOVE_SLOTS = 64              # toy: prefer which square / slot

chess_net = TwoLayerNet(
    input_dim=BOARD_FEATURES,
    hidden_dim=32,
    output_dim=MOVE_SLOTS,
)

fake_board = torch.randn(BOARD_FEATURES)
move_logits = chess_net(fake_board)
move_probs = nn.functional.softmax(move_logits, dim=-1)

print("input", fake_board.shape)
print("logits", move_logits.shape)
print("softmax", move_probs.shape, "sum=", move_probs)
print("most preferred slot:", int(move_probs.argmax()))

input torch.Size([64])
logits torch.Size([64])
softmax torch.Size([64]) sum= tensor([0.0174, 0.0146, 0.0187, 0.0207, 0.0150, 0.0188, 0.0168, 0.0363, 0.0100,
        0.0107, 0.0073, 0.0104, 0.0147, 0.0215, 0.0131, 0.0102, 0.0129, 0.0239,
        0.0106, 0.0194, 0.0124, 0.0107, 0.0128, 0.0083, 0.0206, 0.0149, 0.0212,
        0.0156, 0.0157, 0.0172, 0.0132, 0.0073, 0.0101, 0.0160, 0.0151, 0.0204,
        0.0164, 0.0114, 0.0117, 0.0102, 0.0135, 0.0172, 0.0127, 0.0150, 0.0155,
        0.0091, 0.0135, 0.0149, 0.0165, 0.0251, 0.0168, 0.0271, 0.0138, 0.0143,
        0.0098, 0.0213, 0.0193, 0.0157, 0.0209, 0.0179, 0.0124, 0.0185, 0.0186,
        0.0164], grad_fn=<SoftmaxBackward0>)
most preferred slot: 7


### Go-shaped outputs

Board features in; one score per intersection (+ optional pass). Softmax → preference over board points. Toy board size `5×5` here; full Go is often `19×19 + 1`.

In [13]:
N = 5
GO_FEATURES = N * N
GO_MOVES = N * N + 1         # intersections + pass

go_net = TwoLayerNet(
    input_dim=GO_FEATURES,
    hidden_dim=32,
    output_dim=GO_MOVES,
)

fake_go = torch.randn(GO_FEATURES)
go_logits = go_net(fake_go)
go_probs = nn.functional.softmax(go_logits, dim=-1)

print("input", fake_go.shape)
print("logits", go_logits.shape, "(= board points + pass)")
print("softmax sum", go_probs.sum().item())
choice = int(go_probs.argmax())
if choice == GO_MOVES - 1:
    print("most preferred: pass")
else:
    print("most preferred intersection:", choice, "-> row/col", divmod(choice, N))

input torch.Size([25])
logits torch.Size([26]) (= board points + pass)
softmax sum 0.9999999403953552
most preferred intersection: 6 -> row/col (1, 1)


Compare the three. Pong, chess and Go used the same class, or a near-identical one,
with different numbers for `input_dim` and `output_dim`.

So the difference between "a Pong player" and "a Go player" here is not architecture. It is:

1. how the world is encoded into the input vector,
2. how many outputs there are,
3. and what you decide the outputs mean.

Item 3 costs nothing; it is pure interpretation. Item 1 is where most of the real
engineering difficulty lives, and it is what turns 100,800 raw pixels into 6400 useful
ones.

Note the `+ 1` in `GO_MOVES`. Passing is a legal move in Go, so it needs a slot. Getting
that kind of detail wrong gives you a network that is silently incapable of a legal action,
and nothing in the shapes will warn you.

## 7. Building blocks

### Dropout

Randomly zeros a fraction of units during training so the net does not rely on single neurons. In `model.eval()`, dropout is turned off.

In [14]:
drop_net = TwoLayerNet(input_dim=8, hidden_dim=16, output_dim=4, dropout=0.5)
x = torch.randn(8)

drop_net.train()
y_train = torch.stack([drop_net(x) for _ in range(3)])
print("train mode (stochastic):\n", y_train)

drop_net.eval()
with torch.no_grad():
    y_eval = torch.stack([drop_net(x) for _ in range(3)])
print("eval mode (stable):\n", y_eval)

train mode (stochastic):
 tensor([[ 0.4358,  0.6843, -0.3053,  0.2378],
        [ 0.3018,  0.8393, -0.0768, -0.1456],
        [ 0.1796,  0.8046, -0.0140,  0.1468]], grad_fn=<StackBackward0>)
eval mode (stable):
 tensor([[ 0.2285,  0.6672, -0.1303,  0.0888],
        [ 0.2285,  0.6672, -0.1303,  0.0888],
        [ 0.2285,  0.6672, -0.1303,  0.0888]])


This is the clearest demonstration in the worksheet, so read it carefully. The input
never changed. Yet in train mode the three rows differ from each other, and in eval mode
all three are identical.

That is dropout: in training it randomly switches units off, so the answer is stochastic;
in eval mode it does nothing, so the answer is deterministic.

The practical consequence: forgetting `model.eval()` before evaluating means measuring a
network that is being randomly damaged, and then wondering why your numbers wobble between
runs.

### ReLU vs LeakyReLU

ReLU is $\max(0, z)$. Units that always see negative inputs can get stuck at 0 (“dying ReLU”).

LeakyReLU allows a small slope for negatives so a weak gradient can still flow.

In [15]:
z = torch.tensor([-2.0, -0.5, 0.0, 1.5])
print("z         ", z.tolist())
print("relu      ", nn.functional.relu(z).tolist())
print("leaky_relu", nn.functional.leaky_relu(z, negative_slope=0.01).tolist())

z          [-2.0, -0.5, 0.0, 1.5]
relu       [0.0, 0.0, 0.0, 1.5]
leaky_relu [-0.019999999552965164, -0.004999999888241291, 0.0, 1.5]


Read the two negative entries. ReLU maps both `-2.0` and `-0.5` to exactly `0.0`, the
difference between them is destroyed, and with it any information about *how* negative they
were.

LeakyReLU maps them to small negative numbers instead. Tiny, but the ordering survives, and
so does a gradient. That is the whole argument for LeakyReLU, in one line of output.

### BatchNorm and weight init

BatchNorm (`nn.BatchNorm1d`) renormalizes along the batch for each feature, useful in larger nets.

Init sets starting weights. Common choices: Xavier uniform, or Normal with `std = 1 / sqrt(fan_in)`.

In [16]:
layer = nn.Linear(16, 8)
nn.init.xavier_uniform_(layer.weight)
nn.init.zeros_(layer.bias)
print("xavier weight std~", layer.weight.std().item())

layer2 = nn.Linear(16, 8, bias=False)
nn.init.normal_(layer2.weight, mean=0.0, std=1.0 / math.sqrt(16))
print("fan-in normal std~", layer2.weight.std().item())

# BatchNorm expects (batch, features)
bn = nn.BatchNorm1d(8)
batch = torch.randn(32, 8)
print("batchnorm", batch.shape, "->", bn(batch).shape)

xavier weight std~ 0.29966089129447937
fan-in normal std~ 0.2634861469268799
batchnorm torch.Size([32, 8]) -> torch.Size([32, 8])


The two initialisers do not aim at the same number, and it is worth seeing why. The
fan-in scheme targets exactly $1/\sqrt{16} = 0.25$, which is what we asked for. Xavier
targets $\sqrt{2/(\text{fan-in}+\text{fan-out})} = \sqrt{2/24} \approx 0.29$, because it
accounts for the outputs as well as the inputs.

What they share is the idea: keep a layer's output scale roughly independent of how many
inputs it has. Stack a few layers that each multiply the scale by 3 and the numbers
reaching the end are enormous; make each one shrink it and they vanish. Either way learning
stalls before it starts.

Note `layer2` was built with `bias=False`. The real Pong policy does the same, two bare
weight matrices, no bias terms anywhere.

BatchNorm needs a batch dimension, since it computes statistics across the batch. Hand it a
single unbatched example and it complains. That is one reason this project does not use it:
it processes one frame at a time.

## 8. Device and checkpoints

Put both the model and its inputs on the same device. Save / load weights with `state_dict`.

In [17]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("device:", device)

net = TwoLayerNet(8, 16, 4).to(device)
x = torch.randn(8, device=device)
print("forward on device:", net(x).shape)

ckpt = pong.PROJECT_ROOT / "assets" / "_demo_two_layer.pt"   # not Path("assets"): the notebook is not the root
torch.save(net.state_dict(), ckpt)

restored = TwoLayerNet(8, 16, 4).to(device)
restored.load_state_dict(torch.load(ckpt, map_location=device, weights_only=True))
restored.eval()
print("restored forward:", restored(x).shape)
print("saved to", ckpt)

device: mps


forward on device: torch.Size([4])
restored forward: torch.Size([4])
saved to /Users/piyush/Code/ml/pong/assets/_demo_two_layer.pt


## 9. What to try next

- Change `hidden_dim` (e.g. 8, 32, 128) and watch output shapes stay tied to `output_dim`.
- Pass a batch `(B, input_dim)` into your net and confirm the batch size is preserved.
- Practice `unsqueeze` / `squeeze` until shape errors feel readable.
- Scale inputs to a similar range (e.g. roughly $[0,1]$ or zero-mean) when you work with real data.
- Later, when you train: `loss.backward()` then `optimizer.step()`, not covered here on purpose.

You now have the core PyTorch pieces: Module → instantiate → forward → shapes.

## 10. Your exercises

In [18]:
# Exercise 2.1: Predict, then check.
# Build a TwoLayerNet with input_dim=10, hidden_dim=4, output_dim=2. Before running
# anything, write in a comment what each of these gives:
#   a) a single example of shape (10,)
#   b) a batch of shape (7, 10)
#   c) a wrong input of shape (10, 1)
# Then run all three and see whether you were right. Catch the error for (c).

# your code here

<details>
<summary><b>Answer 2.1</b> (try it first, then click)</summary>

```python
net = TwoLayerNet(input_dim=10, hidden_dim=4, output_dim=2)

# a) one example, no batch axis
print("a)", net(torch.randn(10)).shape)            # -> torch.Size([2])

# b) a batch of seven
print("b)", net(torch.randn(7, 10)).shape)         # -> torch.Size([7, 2])

# c) the wrong way round
try:
    net(torch.randn(10, 1))
except RuntimeError as error:
    print("c) RuntimeError:", error)
```

(a) gives `(2,)` and (b) gives `(7, 2)`: the batch dimension passes through untouched and
only the last dimension changes, from 10 to 2.

(c) fails, and the reason is worth saying precisely. `(10, 1)` is not "ten features
arranged in a column" as far as `Linear` is concerned: it is ten examples with one
feature each. The layer looks at the last dimension, finds 1, and wants 10.

If you actually have one example and need a batch axis, `unsqueeze(0)` gives `(1, 10)`,
which works. `unsqueeze(-1)` gives `(10, 1)`, which does not. Same numbers, and only one of
them means what you intended.

</details>


**Exercise 2.2.** The Pong-shaped network has about 1.28 million parameters at
`hidden=200`. Work out how many are in layer 1 and how many in layer 2. What does the
ratio tell you about which layer dominates, and why?

*Your written answer here.*

<details>
<summary><b>Answer 2.2</b> (try it first, then click)</summary>

```python
hidden = 200
layer1 = 6400 * hidden          # 1,280,000
layer2 = hidden * 1             #       200
print(f"layer 1: {layer1:>9,}  ({layer1 / (layer1 + layer2):.2%})")
print(f"layer 2: {layer2:>9,}")
print(f"total:   {layer1 + layer2:>9,}")
```

Layer 1 holds 1,280,000 of the 1,280,200 parameters, 99.98%. Layer 2 holds 200.

The reason is that the first layer's size is `inputs × hidden`, and the input is 6400 long,
while the second is `hidden × 1`. So the parameter count is dominated entirely by how wide
the *input* is, not by how deep or clever the network is. This network has two layers and
over a million parameters; you could add several more hidden layers of 200 units and barely
change the total.

Two consequences. First, preprocessing is doing real work: cutting the input from 100,800 to
6400 cut the first layer, and therefore the whole network, by a factor of about 16.
Second, "1.28 million parameters" sounds like a large model and is not: it is a shallow model
with a wide mouth.

</details>


In [19]:
# Exercise 2.3: Count without building.
# Write count_params(input_dim, hidden_dim, output_dim, bias=True) that returns the
# total number of learnable numbers by arithmetic alone. Check it against a real
# TwoLayerNet at three different sizes.

# your code here

<details>
<summary><b>Answer 2.3</b> (try it first, then click)</summary>

```python
def count_params(input_dim, hidden_dim, output_dim, bias=True):
    weights = input_dim * hidden_dim + hidden_dim * output_dim
    biases = (hidden_dim + output_dim) if bias else 0
    return weights + biases


for dims in [(8, 5, 3), (16, 8, 4), (6400, 200, 1)]:
    predicted = count_params(*dims)
    actual = sum(p.numel() for p in TwoLayerNet(*dims).parameters())
    print(f"{str(dims):>18}  predicted {predicted:>9,}  actual {actual:>9,}  "
          f"{'match' if predicted == actual else 'MISMATCH'}")

# And the no-bias case, which is what the real policy uses:
no_bias = count_params(6400, 200, 1, bias=False)
print(f"\nwithout biases: {no_bias:,}")
```

All three should match. `(6400, 200, 1)` gives 1,280,401 with biases and 1,280,200
without, and that second number is the one you saw for the real `Policy`, which is built
with `bias=False`.

The arithmetic is worth being able to do in your head, because it tells you whether a
network will fit in memory before you build it. Weights are `in × out` per layer; biases are
one per output unit, which is almost always negligible by comparison.

</details>


In [20]:
# Exercise 2.4: Make the dying ReLU visible.
# Feed an all-negative input through a Linear layer whose weights you set to a positive
# constant, then through F.relu, and show the output is all zeros. Show F.leaky_relu is
# not. Then say what that would do to learning.

# your code here

<details>
<summary><b>Answer 2.4</b> (try it first, then click)</summary>

```python
layer = nn.Linear(4, 3)
with torch.no_grad():
    layer.weight.fill_(0.5)       # all positive weights
    layer.bias.fill_(0.0)

all_negative = torch.tensor([-1.0, -2.0, -3.0, -4.0])
pre_activation = layer(all_negative)

print("before activation:", pre_activation)
print("relu:             ", F.relu(pre_activation))
print("leaky_relu:       ", F.leaky_relu(pre_activation, negative_slope=0.01))

# The part that actually matters: the gradient.
for name, activation in [("relu", F.relu), ("leaky_relu", F.leaky_relu)]:
    # detach() first: pre_activation came out of a layer, so it is not a leaf
    # tensor, and .grad is only populated on leaves.
    x = pre_activation.detach().clone().requires_grad_(True)
    activation(x).sum().backward()
    print(f"{name:11s} gradient: {x.grad.tolist()}")
```

Positive weights times an all-negative input give an all-negative pre-activation, so ReLU
returns exactly zeros and LeakyReLU returns small negative numbers.

But the output is not the real problem, the gradient is. ReLU's gradient is zero
everywhere the input was negative, so `backward()` sends nothing back through that unit.
Its weights receive no update, so its input keeps being negative, so its gradient stays
zero. The unit is dead, permanently, and no amount of further training revives it. That is
the "dying ReLU".

LeakyReLU's gradient is 0.01 rather than 0. Tiny, but nonzero, so the unit can still be
pushed back toward usefulness.

Why this network gets away with plain ReLU: it has one hidden layer, so a dead unit costs
you 1/200th of the model's capacity rather than breaking a deep chain.

</details>


**Exercise 2.5.** The same class served as a Pong, chess and Go player. Pick a game or task
you know well and describe: what goes into the input vector, how long is it, how many
outputs do you need, and what does each output mean? Be specific about the encoding, that
is the hard part.

*Your written answer here.*

<details>
<summary><b>Answer 2.5</b> (try it first, then click)</summary>

There is no single right answer, so here is one worked example and the trap to avoid.

Task: predicting whether a chess position is winning for white.

- Input. The obvious encoding is 64 squares × 12 piece types (6 per colour), one-hot,
  giving 768 numbers. Add a few for castling rights, whose turn it is, and the
  fifty-move counter, say 776.
- Output. One number, through a sigmoid: P(white wins). One output, because the
  question has one answer.
- Meaning. Above 0.5 means white is better. Nothing in the network knows that; it is
  our reading, exactly as with `P(UP)` for Pong.

The trap, and the reason encoding is the hard part: it is tempting to feed the network
64 numbers, one per square, with pieces as integers, 1 for pawn, 5 for rook, 9 for queen.
That encoding quietly asserts a queen is nine pawns *and* that a rook is numerically
between a bishop and a queen in a way the network should interpolate through. One-hot
asserts nothing beyond "these are different things", which is usually what you want.

The general question to ask of any encoding: what relationships am I implying between
these numbers, and are they true? Pong's binarised frame implies only "pixel occupied or
not", which is honest. Raw RGB would imply that colour distances mean something.

</details>


In [21]:
# Exercise 2.6: A round trip that fails.
# Save a TwoLayerNet(8, 16, 4) state_dict, then try to load it into a
# TwoLayerNet(8, 32, 4). Catch the error and read it. Which numbers does it name, and
# what does that tell you about what a checkpoint does and does not contain?

# your code here

<details>
<summary><b>Answer 2.6</b> (try it first, then click)</summary>

```python
saved = TwoLayerNet(8, 16, 4)
torch.save(saved.state_dict(), OUT / "mismatch_demo.pt")

wrong_shape = TwoLayerNet(8, 32, 4)      # 32 hidden units, not 16
try:
    wrong_shape.load_state_dict(torch.load(OUT / "mismatch_demo.pt", weights_only=True))
except RuntimeError as error:
    print("RuntimeError:\n", error)
```

The error names the mismatched shapes for each parameter: it expected `(32, 8)` for
`layer1.weight` and found `(16, 8)`, and likewise for the other tensors.

What that tells you is what a checkpoint is: an ordered mapping from parameter name to
tensor, and nothing else. No architecture, no class, no record of the hyperparameters. To
load one you must first construct the exact network that produced it, which means you need
to know `hidden=16` from somewhere outside the file.

This is a real failure mode rather than a toy one. It is why `scripts/train.py` stores its
settings *inside* the checkpoint, and why resuming with a different `--hidden` is refused
outright with a message naming the right value, a loud failure instead of a confusing one.

Note also that a mismatch this size fails loudly. A mismatch that happens to have
compatible shapes would load silently and give you a network whose weights mean something
else entirely.

</details>


## 11. Questions to think about

1. `model.parameters()` only finds layers assigned to `self`. What kind of bug does that
   create, would anything visibly fail, or would training quietly do less than you think?

2. The Pong network puts 1.28 million parameters in its first layer purely because the input
   is 6400 long. If you preprocessed the frame down to 20 numbers first, the network would
   be tiny. So why do people train on pixels at all?

3. Dropout makes training stochastic on purpose, the way sticky actions did in Worksheet 1.
   What do those two ideas have in common?

4. We read one sigmoid output as "probability of UP". Suppose you used two outputs and a
   softmax instead. Would that be a different agent, or the same one written differently?

5. A `state_dict` stores weights and not architecture. What would it take to make a
   checkpoint genuinely self-describing, and what could still go wrong?

## 12. What you have learnt

| You can now | Concretely |
|---|---|
| Read a diagram as code | inputs → `Linear(in, hidden)` → hidden → `Linear(hidden, out)` |
| Reason about shapes | `Linear` applies to the last dim; batch dims pass through |
| Read a shape error | map "mat1 and mat2" back onto your own last dimension |
| Define a network | subclass, `super().__init__()`, register layers, `forward` |
| Find what is learnable | `named_parameters()`, and why unregistered layers are invisible |
| Turn outputs into decisions | sigmoid for one probability, softmax for a distribution |
| See the interpretation layer | the same net is Pong, chess or Go depending on what you call the numbers |
| Use the standard blocks | dropout and `train()`/`eval()`, ReLU vs LeakyReLU, BatchNorm, init |
| Save and restore | `state_dict`, `map_location`, `weights_only`, and why architecture must match |

What is still missing is learning. Every network here was never told whether its answers
were any good.


## Next

Continue to **Worksheet 3: one training step**, which is where the network finally
gets told whether its answers were any good.

[Open Worksheet 3 in Colab](https://colab.research.google.com/github/piyushahuja/pong/blob/main/notebooks/03_train_step.ipynb)
